In [2]:
import pandas as pd
df = pd.read_csv('eia_data.csv')

# Filter to only years 1981 and later
df = df[df['Year'] >= 1981]
df = df[df['State'] != 'US']  

# Add Region
region_mapping = {
    'CT': 'Northeast', 'ME': 'Northeast', 'MA': 'Northeast', 'NH': 'Northeast',
    'RI': 'Northeast', 'VT': 'Northeast', 'NJ': 'Northeast', 'NY': 'Northeast', 'PA': 'Northeast',
    'DE': 'South', 'FL': 'South', 'GA': 'South', 'MD': 'South', 'NC': 'South',
    'SC': 'South', 'VA': 'South', 'WV': 'South', 'AL': 'South', 'KY': 'South',
    'MS': 'South', 'TN': 'South', 'AR': 'South', 'LA': 'South', 'OK': 'South', 
    'TX': 'South', 'DC': 'South',
    'IL': 'Midwest', 'IN': 'Midwest', 'MI': 'Midwest', 'OH': 'Midwest', 'WI': 'Midwest',
    'IA': 'Midwest', 'KS': 'Midwest', 'MN': 'Midwest', 'MO': 'Midwest', 'NE': 'Midwest',
    'ND': 'Midwest', 'SD': 'Midwest',
    'AZ': 'West', 'CO': 'West', 'ID': 'West', 'MT': 'West', 'NV': 'West',
    'NM': 'West', 'UT': 'West', 'WY': 'West', 'AK': 'West', 'CA': 'West',
    'HI': 'West', 'OR': 'West', 'WA': 'West'
}

df['Region'] = df['State'].map(region_mapping)

# Add Dominant Energy Source
def get_dominant_energy(row):
    sources = {
        'Coal': row['Coal_Consumption'],
        'Natural Gas': row['NatGas_Consumption'],
        'Nuclear': row['Nuclear_Consumption'],
        'Renewables': row['Renewable_Consumption']
    }
    return max(sources, key=sources.get)

df['Dominant_Energy_Source'] = df.apply(get_dominant_energy, axis=1)

len(df)

1377

In [3]:
import altair as alt
import pandas as pd

# data processing
df_bar = pd.read_csv('eia_data.csv')
df_bar = df_bar[(df_bar['Year'] >= 1981) & (df_bar['State'] != 'US')]
df_bar['Region'] = df_bar['State'].map(region_mapping)
df_bar = df_bar.dropna(subset=['Region'])

# Average CO2 per state across all years
avg_by_state = (
    df_bar.groupby(['State', 'Region'])
    .agg(
        Avg_CO2=('CO2_Emissions', 'mean'),
        Avg_Energy=('Total_Energy_Consumption', 'mean'),
        Years=('Year', lambda x: f"{int(x.min())}\u2013{int(x.max())}")
    )
    .reset_index()
)

# region filter
region_select = alt.selection_point(
    fields=['Region'],
    bind='legend',
    name='region_filter'
)

# color scale
color_scale = alt.Scale(
    domain=['Northeast', 'South', 'Midwest', 'West'],
    range=['#3b6ea5', '#c0552a', '#4a9a6e', '#8b5ea5']
)

# chart
bar = (
    alt.Chart(avg_by_state)
    .mark_bar(cornerRadiusTopLeft=3, cornerRadiusTopRight=3)
    .encode(
        x=alt.X(
            'State:N',
            sort=alt.EncodingSortField(field='Avg_CO2', order='descending'),
            title='State',
            axis=alt.Axis(labelAngle=-45, labelFontSize=10)
        ),
        y=alt.Y(
            'Avg_CO2:Q',
            title='Avg CO\u2082 Emissions (Million Metric Tons)',
            axis=alt.Axis(grid=True)
        ),
        color=alt.Color(
            'Region:N',
            scale=color_scale,
            legend=alt.Legend(
                title='Region (click to filter)',
                orient='top-right',
                symbolType='circle'
            )
        ),
        opacity=alt.condition(region_select, alt.value(0.85), alt.value(0.1)),
        tooltip=[
            alt.Tooltip('State:N',      title='State'),
            alt.Tooltip('Region:N',     title='Region'),
            alt.Tooltip('Avg_CO2:Q',    title='Avg CO\u2082 (Mil. Mt)',    format='.1f'),
            alt.Tooltip('Avg_Energy:Q', title='Avg Energy (B. Btu)', format=',.0f'),
            alt.Tooltip('Years:N',      title='Year range'),
        ]
    )
    .add_params(region_select)
    .properties(
        width=680,
        height=380,
        title=alt.TitleParams(
            text='Average CO\u2082 Emissions by State (1981\u20132023)',
            subtitle='Sorted by emissions \u00b7 Click a region in the legend to filter \u00b7 Hover for details',
            fontSize=15,
            subtitleFontSize=11,
            subtitleColor='#6b6560'
        )
    )
    .configure_axis(
        labelFont='sans-serif',
        titleFont='sans-serif',
        titleFontSize=12,
        gridColor='#e8e4df'
    )
    .configure_view(strokeWidth=0)
)

bar


/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.Chart(...)

In [4]:
import altair as alt
import pandas as pd

regions = ['Midwest', 'Northeast', 'South', 'West']

data = pd.DataFrame([
    {'Region': r, 'Source': s, 'Share': v, 'Abs': a}
    for r, coal, gas, nuc, ren, ac, ag, an, ar in [
        ('Midwest',   42.30, 36.22, 12.62,  8.86, 556696, 476617, 166021, 116624),
        ('Northeast', 18.06, 49.25, 23.35,  9.33, 159576, 435162, 206312,  82450),
        ('South',     30.76, 46.46, 14.41,  8.37, 449986, 679733, 210810, 122450),
        ('West',      23.24, 53.71,  7.12, 15.94, 171015, 395218,  52380, 117271),
    ]
    for s, v, a in [
        ('Coal', coal, ac),
        ('Natural gas', gas, ag),
        ('Nuclear', nuc, an),
        ('Renewables', ren, ar),
    ]
])

color_scale = alt.Scale(
    domain=['Coal', 'Natural gas', 'Nuclear', 'Renewables'],
    range=['#73726c', '#3266ad', '#8b5ea5', '#4a9a6e']
)

order = ['Coal', 'Natural gas', 'Nuclear', 'Renewables']

chart = (
    alt.Chart(data)
    .mark_bar()
    .encode(
        x=alt.X('Region:N', title=None, axis=alt.Axis(labelAngle=0)),
        y=alt.Y('Share:Q', title='Share of energy mix (%)', stack='zero', scale=alt.Scale(domain=[0, 100])),
        color=alt.Color('Source:N', scale=color_scale,
                        sort=order,
                        legend=alt.Legend(title='Source', orient='top')),
        order=alt.Order('Source:N', sort='ascending'),
        tooltip=[
            alt.Tooltip('Region:N', title='Region'),
            alt.Tooltip('Source:N', title='Source'),
            alt.Tooltip('Share:Q', title='Share (%)', format='.1f'),
            alt.Tooltip('Abs:Q', title='Avg (bil. Btu)', format=',.0f'),
        ]
    )
    .properties(
        width=500,
        height=340,
        title=alt.TitleParams(
            text='Average energy mix by region (1981–2023)',
            subtitle='Share of Coal + Natural Gas + Nuclear + Renewable consumption',
            fontSize=15,
            subtitleFontSize=11,
            subtitleColor='#6b6560'
        )
    )
    .configure_axis(labelFont='sans-serif', titleFont='sans-serif', titleFontSize=12, gridColor='#e8e4df')
    .configure_view(strokeWidth=0)
)

chart

/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.Chart(...)

In [5]:
bar.save("summary.html")
chart.save("stacked_bar.html")

/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/c